In [1]:
debugging_mode=True
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window

import ConnectionConfig as cc
debugging_mode=True
cc.setupEnvironment()


Environment variables are set...
Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [2]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
spark = cc.startLocalCluster("DIM_DATE",4)
spark.getActiveSession()

In [4]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [5]:
#get info
#
date_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select id, log_type, log_time from treasure_log) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
    .filter(col("log_type") == 2)
)


In [6]:
#Haal unieke datums uit log_time
neededDates = (
    date_src
    .withColumn("calendarDate", to_date(col("log_time")))
    .select("calendarDate")
    .distinct()
    .orderBy("calendarDate")
)

In [7]:

windowSpec = Window.orderBy("calendarDate")

dimDate = (
    neededDates
    .withColumn("DateSurKey", expr("uuid()"))  # UUID per rij
    .withColumn("DateId", row_number().over(windowSpec))  # Oplopende ID
    .withColumn("Day", dayofmonth(col("calendarDate")))
    .withColumn("Week", weekofyear(col("calendarDate")))
    .withColumn("Month", date_format(col("calendarDate"), "MMMM"))
    .withColumn("Year", year(col("calendarDate")))
    .withColumn("MonthOfTheYear", month(col("calendarDate")))
    .withColumn("DayOfTheWeek", weekday(col("calendarDate")) + 1)  # 1 = maandag, 7 = zondag
    .withColumn("IsWeekDay", when(weekday(col("calendarDate")) < 5, True).otherwise(False))
    .select(
        "DateSurKey",
        "DateId",
        "Day",
        "Week",
        "Month",
        "Year",
        "MonthOfTheYear",
        "DayOfTheWeek",
        "IsWeekDay"
    )
)

dimDate.show()


+--------------------+------+---+----+---------+----+--------------+------------+---------+
|          DateSurKey|DateId|Day|Week|    Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|
+--------------------+------+---+----+---------+----+--------------+------------+---------+
|58ce0efc-8cbb-470...|     1| 11|  37|September|2020|             9|           5|     true|
|ded156f9-31e5-4d8...|     2| 12|  37|September|2020|             9|           6|    false|
|7bec5d27-8650-408...|     3| 13|  37|September|2020|             9|           7|    false|
|6fdeed2f-256d-42d...|     4| 14|  38|September|2020|             9|           1|     true|
|dbf041c0-7414-42c...|     5| 15|  38|September|2020|             9|           2|     true|
|0d1de084-eda8-43b...|     6| 16|  38|September|2020|             9|           3|     true|
|c4bf402d-7e5d-49a...|     7| 17|  38|September|2020|             9|           4|     true|
|075f840d-ed50-452...|     8| 18|  38|September|2020|             9|           5

In [8]:
#opslagen tabel
dimDate.write.format("delta").mode("overwrite").save("delta/DATE_DIM")


In [9]:
spark.stop()